# 01 — Exploração dos dados

**Projeto:** Health Intelligence Lab — Análise de desempenho e sinistralidade em
operadoras de Autogestão

**Objetivo deste notebook:** entender a estrutura, granularidade e conteúdo de
cada base pública da ANS antes de qualquer tratamento, e registrar as decisões
tomadas a partir dessa exploração inicial.

**Fontes utilizadas:**

| Base | Arquivo | Publicação |
|---|---|---|
| Demonstrações Contábeis (DIOPS) | `1T2025.csv` … `1T2026.csv` | ANS — Dados Abertos |
| Relação de Operadoras Ativas | `Relatorio_cadop.csv` | ANS — Dados Abertos |
| Caderno de Informação da Saúde Suplementar | `caderno_jun26.ods` | ANS |

Os arquivos brutos não são versionados neste repositório (ver `.gitignore` e a
seção *Como reproduzir* do README) por serem grandes (dezenas de MB cada). O
notebook assume que eles estão em `data/raw/`.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

RAW = '../data/raw'


## 1. Demonstrações Contábeis (DIOPS)

Arquivo trimestral publicado pela ANS com o balancete de cada operadora,
detalhado por conta contábil (plano de contas padrão do setor).


In [2]:
dio = pd.read_csv(f'{RAW}/1T2026.csv', sep=';', decimal=',', encoding='utf-8')
print("Shape:", dio.shape)
dio.head()


Shape: (702622, 6)


,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2026-01-01,350648,463119019,Outras Despesas,0.0,0.00
1,2026-01-01,350648,46311911,Despesas com Localização e Manutenção,0.0,37265.23
2,2026-01-01,350648,463119111,Aluguel,0.0,3481.67
3,2026-01-01,350648,463119113,Condomínios e Taxas,0.0,253.47
4,2026-01-01,350648,463119114,"Água, Luz e Gás",0.0,8184.90


In [3]:
print("Colunas:", dio.columns.tolist())
print("\nTipos:")
print(dio.dtypes)


Colunas: ['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Tipos:
DATA                     str
REG_ANS                int64
CD_CONTA_CONTABIL      int64
DESCRICAO                str
VL_SALDO_INICIAL     float64
VL_SALDO_FINAL       float64
dtype: object


In [4]:
print("Trimestre de referência (coluna DATA):", dio['DATA'].unique())
print("Operadoras distintas (REG_ANS):", dio['REG_ANS'].nunique())
print("Contas contábeis distintas:", dio['CD_CONTA_CONTABIL'].nunique())


Trimestre de referência (coluna DATA): <StringArray>
['2026-01-01']
Length: 1, dtype: str
Operadoras distintas (REG_ANS): 781
Contas contábeis distintas: 7220


### 1.1 Granularidade e hierarquia do plano de contas

O campo `CD_CONTA_CONTABIL` é hierárquico: o número de dígitos indica o nível
de detalhamento (contas de 1 dígito são as mais agregadas — Receitas e
Despesas totais — e contas de 9 dígitos são as folhas mais detalhadas).


In [5]:
dio['cd_str'] = dio['CD_CONTA_CONTABIL'].astype(str)
dio['nivel'] = dio['cd_str'].str.len()
dio.groupby('nivel')['cd_str'].nunique().rename('contas_distintas_no_nivel')


nivel
1       6
2      23
3      75
4     194
5     301
6     347
8    1562
9    4712
Name: contas_distintas_no_nivel, dtype: int64

In [6]:
# Contas de nível 1 e 2 — a visão macro do balancete (usada como referência)
nivel_macro = dio[dio['nivel'] <= 2][['CD_CONTA_CONTABIL', 'DESCRICAO']].drop_duplicates()
nivel_macro.sort_values('CD_CONTA_CONTABIL')


,CD_CONTA_CONTABIL,DESCRICAO
3892,1,ATIVO
1095,2,PASSIVO
2236,3,RECEITAS
265,4,DESPESAS
161,6,CONTAS DE DESTINAÇÃO/APURAÇÃO DE RESULTADO
185,7,CONTAS TRANSITÓRIAS - APURAÇÃO DE CUSTOS
3893,12,ATIVO CIRCULANTE
3064,13,ATIVO NÃO CIRCULANTE
17186,19,COMPENSAÇÃO - ATIVO
1096,21,PASSIVO CIRCULANTE


**Contas-chave identificadas para a análise:**

| Conta | Descrição | Uso |
|---|---|---|
| `3` | Receitas totais | Resultado operacional completo |
| `4` | Despesas totais | Resultado operacional completo |
| `31` | Receitas com Operações de Assistência à Saúde | Denominador da sinistralidade |
| `41` | Eventos Indenizáveis Líquidos / Sinistros Retidos | Numerador da sinistralidade |
| `411` | Eventos/Sinistros Conhecidos ou Avisados | Base para decomposição de glosa |
| `46` | Despesas Administrativas | Indicador de eficiência operacional |

A composição de faturado bruto / glosa / coparticipação dentro da conta `411`
é explorada no notebook `03_pasa_analysis`.

### 1.2 Um ponto de atenção identificado nesta exploração

Os valores de `VL_SALDO_FINAL` na DIOPS são **acumulados no ano-calendário**,
não isolados por trimestre — confirmado ao observar que a receita de uma
mesma operadora cresce de forma consistente com o avanço dos trimestres dentro
de 2025, resetando no início de 2026. Esse comportamento é tratado no notebook
`02_data_quality`.


In [7]:
# Evidência do comportamento acumulado: receita total (conta 3) de uma operadora
# ao longo dos arquivos trimestrais de 2025 deveria crescer monotonicamente
# dentro do mesmo ano se for acumulado.
amostra_reg_ans = dio['REG_ANS'].iloc[0]

valores = {}
for trimestre, arquivo in [('1T2025', '1T2025.csv'), ('2T2025', '2T2025.csv'),
                            ('3T2025', '3T2025.csv'), ('4T2025', '4T2025.csv')]:
    d = pd.read_csv(f'{RAW}/{arquivo}', sep=';', decimal=',', encoding='utf-8')
    linha = d[(d['REG_ANS'] == amostra_reg_ans) & (d['CD_CONTA_CONTABIL'] == 3)]
    if len(linha):
        valores[trimestre] = linha['VL_SALDO_FINAL'].values[0]

pd.Series(valores, name='receita_total_acumulada')


1T2025    13622103.62
2T2025    27945969.10
3T2025    42282293.90
4T2025    57943489.63
Name: receita_total_acumulada, dtype: float64

## 2. Relação de Operadoras Ativas (cadastro)

Cadastro com a identificação, modalidade e localização de cada operadora
ativa. Chave de cruzamento com a DIOPS: `REGISTRO_OPERADORA` ↔ `REG_ANS`.


In [8]:
cad = pd.read_csv(f'{RAW}/Relatorio_cadop.csv', sep=';', encoding='utf-8', dtype=str)
print("Shape:", cad.shape)
cad.head()


Shape: (1111, 20)


,REGISTRO_OPERADORA,CNPJ,RAZAO_SOCIAL,NOME_FANTASIA,MODALIDADE,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,UF,CEP,DDD,TELEFONE,FAX,ENDERECO_ELETRONICO,REPRESENTANTE,CARGO_REPRESENTANTE,REGIAO_DE_COMERCIALIZACAO,DATA_REGISTRO_ANS
0,419761,19541931000125,18 DE JULHO ADMINISTRADORA DE BENEFÍCIOS LTDA,NaN,Administradora de Benefícios,RUA CAPITÃO MEDEIROS DE REZENDE,274,NaN,PRAÇA DA BANDEIRA,Além Paraíba,MG,36660000,32,34624649,NaN,contabilidade@cbnassessoria.com.br,LUIZ HENRIQUE MARENDINO GONÇALVES,SÓCIO ADMINISTRADOR,6,2015-05-19
1,421545,22869997000153,2B ODONTOLOGIA OPERADORA DE PLANOS ODONTOLÓGIC...,NaN,Odontologia de Grupo,RUA CATÃO,128,SALA 126,VILA ROMANA,São Paulo,SP,05049000,11,34415852,NaN,labmarisol@gmail.com,MARISOL BECHELLI,SÓCIO ADMINISTRADORA,4,2019-06-13
2,421421,27452545000195,2CARE OPERADORA DE SAÚDE LTDA.,NaN,Medicina de Grupo,RUA: BERNARDINO DE CAMPOS,230,1º ANDAR,CENTRO,Campinas,SP,13010151,19,37901224,NaN,ans.plano@hospitalcare.com.br,PEDRO REGISTRO MESQUITA,DIRETOR,4,2018-10-09
3,418030,13138885000131,A.P.S. ADMINISTRADORA DE BENEFÍCOS LTDA.,A.P.S. SAÚDE.,Administradora de Benefícios,RUA VOLUNTÁRIOS DA PÁTRIA,2525,CONJUNTO 143 - SALA 01,SANTANA,São Paulo,SP,02401000,11,45223468,NaN,diretoria@apssaude.com.br,NATALIA GAETA NOVAES,SóCIA-ADMINISTRADORA E REPRESENTANTE,4,2011-05-05
4,314668,17505793000101,ABERTTA SAÚDE - ASSOCIAÇÃO BENEFICENTE DOS EMP...,ABERTTA SAÚDE,Autogestão,AV. BERNARDO MONTEIRO,831,"Subsolo, 2º andar e 3º andar",SANTA EFIGÊNIA,Belo Horizonte,MG,30150281,31,32484300,32484377,abertta.ans@arcelormittal.com.br,WERNER DUARTE DALLA,Diretor Presidente,4,1998-12-28


In [9]:
cad['MODALIDADE'].value_counts()


MODALIDADE
Cooperativa Médica                   262
Medicina de Grupo                    253
Administradora de Benefícios         183
Odontologia de Grupo                 142
Autogestão                           142
Cooperativa odontológica              89
Filantropia                           33
Seguradora Especializada em Saúde      7
Name: count, dtype: int64

O universo deste estudo é o das operadoras classificadas como **Autogestão** —
modalidade da PASA e o segmento de comparação natural para o estudo de caso.


In [10]:
autogestoes = cad[cad['MODALIDADE'] == 'Autogestão'].copy()
print(f"Operadoras de Autogestão no cadastro: {len(autogestoes)}")
autogestoes[['REGISTRO_OPERADORA', 'RAZAO_SOCIAL', 'UF']].head()


Operadoras de Autogestão no cadastro: 142


,REGISTRO_OPERADORA,RAZAO_SOCIAL,UF
4,314668,ABERTTA SAÚDE - ASSOCIAÇÃO BENEFICENTE DOS EMP...,MG
14,368920,AGROS - INSTITUTO UFV DE SEGURIDADE SOCIAL,MG
41,423319,ANAFE SAUDE,GO
46,416070,ARCELORMITTAL BRASIL S/A,MG
54,423432,ASSOC DE ASSIST À SAÚDE DOS SERV DAS UNIV E IN...,PE


### 2.1 Cruzamento com a DIOPS

Nem toda operadora do cadastro tem demonstração contábil no trimestre (pode
não ter reportado ainda, ou ser uma Administradora de Benefícios sem
demonstração assistencial própria).


In [11]:
reg_dio = set(dio['REG_ANS'].unique())
reg_autogestoes = set(autogestoes['REGISTRO_OPERADORA'].astype(int))

interseccao = reg_dio & reg_autogestoes
print(f"Autogestões no cadastro: {len(reg_autogestoes)}")
print(f"Autogestões com demonstração contábil no 1T2026: {len(interseccao)}")
print(f"Autogestões sem demonstração contábil no 1T2026: {len(reg_autogestoes - reg_dio)}")


Autogestões no cadastro: 142
Autogestões com demonstração contábil no 1T2026: 116
Autogestões sem demonstração contábil no 1T2026: 26


Esse número (autogestões com dado disponível) varia por trimestre — a
quantidade exata usada em cada período, e os critérios adicionais de
qualidade aplicados, são documentados e validados no notebook
`02_data_quality`.


## 3. Caderno de Informação da Saúde Suplementar

Conjunto de tabelas agregadas (nível Brasil) publicadas pela ANS. Diferente da
DIOPS, aqui o dado já vem consolidado por modalidade, UF ou faixa etária —
não por operadora individual.


In [12]:
xl = pd.ExcelFile(f'{RAW}/caderno_jun26.ods', engine='odf')
abas_tabela = [s for s in xl.sheet_names if s.startswith('tab')]
print(f"Total de abas: {len(xl.sheet_names)} ({len(abas_tabela)} são tabelas de dados)")


Total de abas: 35 (16 são tabelas de dados)


In [13]:
# Tabela 10: Receitas e despesas por modalidade — a mais relevante para
# contextualizar o segmento de Autogestão frente às demais modalidades.
tab10 = xl.parse('tab_10', header=None)
tab10.head(15)


,0,1,2,3,4,5,6,7
0,"Tabela 10 - Receitas e despesas, por modalida...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(Brasil - 1º trimestre/2026),NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Modalidade da operadora,Receita de contraprestações,Outras receitas operacionais,Despesa assistencial,Despesa administrativa,Despesa de comercialização,Outras despesas operacionais,Receitas Administrativas
4,Total,88806060542,5768871924,70622671994,8171061372,3227455194,8758761594,8658014
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Operadoras médico-hospitalares,86953119295,5736540584,70191310182,7615015456,2921711234,8558048847,8276781
7,Autogestão,9217785175,910289020,7967795865,894763370,11833086,843316981,127929
8,Cooperativa Médica,28485538564,2014698324,22214836868,3008094025,341856597,3465138050,6111281
9,Filantropia,1193129755,1392062420,915751315,115873781,18065259,1501533462,0


Esta tabela é usada no notebook `03_pasa_analysis` para situar a sinistralidade
consolidada do segmento de Autogestão frente a outras modalidades do mercado.

## Próximos passos

O notebook `02_data_quality.ipynb` documenta e trata, de forma explícita:
- a conversão de valores acumulados para valores trimestrais isolados;
- os critérios de exclusão/sinalização de operadoras com dado pouco confiável;
- a variação no número de operadoras disponíveis entre trimestres;
- a validação cruzada dos resultados.
